# 4. Relaxar integridade e flexibilizar metas são operações diferentes

In [1]:
from pathlib import Path
import sys, json, numpy as np
ROOT = Path.cwd().resolve()
if not (ROOT / 'optimization_examples').is_dir():
    ROOT = ROOT.parent  # notebooks opened from codes/notebooks/
if not (ROOT / 'optimization_examples').is_dir():
    raise RuntimeError('Abra o notebook a partir do repositório codes ou codes/notebooks.')
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'results' / 'sequence'
OUT.mkdir(parents=True, exist_ok=True)
def save(name, value):
    (OUT / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')


Relaxação linear: trocar x binária por 0 ≤ x ≤ 1. Para MAXIMIZAÇÃO, o ótimo LP é um limite SUPERIOR. Uma fração de kit não é uma entrega válida. Arredondar pode perder qualidade ou violar limites.

In [2]:
from optimization_examples.selection import small_instance, stress_instance, solve, audit
small_lp = solve(small_instance(), relax=True, log_path=OUT/'small-lp.log')
large_lp = solve(stress_instance(), relax=True, log_path=OUT/'stress-lp.log')
for nome, res in [('pequena',small_lp),('estresse',large_lp)]:
    print(nome, res['status'], 'limite superior:', round(res['objective'],3), 'fracionárias:', res['fractional_variables'])
rounded = np.rint(small_lp['x'])
print('Arredondamento ao inteiro mais próximo:', rounded, audit(small_instance(), rounded))
save('relaxation.json', dict(small=small_lp, stress=large_lp))

pequena LP ótimo limite superior: 36.385 fracionárias: 2
estresse LP ótimo limite superior: 5681.626 fracionárias: 9
Arredondamento ao inteiro mais próximo: [0. 1. 0. 1. 1. 1.] {'feasible': True, 'objective': 36.0, 'used': [15.0, 11.0], 'capacity': [15.0, 12.0]}


Diagnóstico separado: se nem LP é viável, o modelo inteiro também é inviável. Se LP é viável, o inteiro ainda pode ser inviável. Não flexibilizar peso físico; só uma meta declarada negociável.

In [3]:
import pulp as pl
from optimization_examples.selection import small_instance, build_model
instance = small_instance()
hard, x = build_model(instance)
hard += pl.lpSum(v*z for v,z in zip(instance['value'],x)) >= 50, 'meta_prioridade'
hard.solve(pl.PULP_CBC_CMD(msg=False))
print('Meta >= 50 com as capacidades originais:',pl.LpStatus[hard.status])
soft, x = build_model(instance)
shortfall = pl.LpVariable('deficit_meta',lowBound=0)
soft += pl.lpSum(v*z for v,z in zip(instance['value'],x)) + shortfall >= 50
# Nova política: minimizar déficit; as capacidades permanecem obrigatórias.
soft.sense = pl.LpMinimize
soft.setObjective(shortfall)
soft.solve(pl.PULP_CBC_CMD(msg=False))
print('Meta flexibilizada:',pl.LpStatus[soft.status],'déficit:',shortfall.value())
print('É solução de OUTRO modelo, que permite déficit de prioridade.')
save('soft-target.json',dict(hard_status=pl.LpStatus[hard.status],soft_status=pl.LpStatus[soft.status],shortfall=shortfall.value()))

Meta >= 50 com as capacidades originais: Infeasible
Meta flexibilizada: Optimal déficit: 14.0
É solução de OUTRO modelo, que permite déficit de prioridade.


In [4]:
from optimization_examples.menu_decision import all_menus
menus = all_menus()
full = [m for m in menus if m['adequacy']==100]
print('Menus possíveis:',len(menus),'que cumprem as três metas:',len(full))
print('Maior fração de metas atingidas:',max(m['adequacy'] for m in menus))
print('Flexibilizar metas muda o problema; não certifica a viabilidade do modelo original.')

Menus possíveis: 81 que cumprem as três metas: 0
Maior fração de metas atingidas: 66.66666666666666
Flexibilizar metas muda o problema; não certifica a viabilidade do modelo original.
